# 03: EDA and Feature Engineering, PD Model

This notebook covers the exploratory data analysis and feature engineering for the PD model features. The analysis and every fitted treatment parameter use the development sample (matured loans, see `01_data_diagnosis`); the treatment is then applied to the whole portfolio, which is scored in `05_training_pd`.

**Input:** `data/processed/checkpoint_DataFrame_for_EDA.pkl`, `data/processed/features_PD_for_EDA.pkl`  
**Output:** `data/processed/checkpoint_df_PD_training.pkl` (treated features for every loan, with the `is_matured` flag), `data/processed/features_PD_num_for_training.pkl`, `data/processed/features_PD_cat_for_training.pkl`

## Libraries

In [ ]:
# Run from the repository root so that data/, models/ and src/ resolve the same way
# whether the notebook is opened from the root or from production_notebooks/.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "production_notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

from feature_plot import plot_boxplots_list, plot_categoricals
from feature_statistics import dataset_audit, outlier_summary
from feature_selection import compute_plot_correlation_matrix
from clean_features_PD import fit_treatment_PD

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

with open("data/processed/features_PD_for_EDA.pkl", "rb") as f:
    features_PD = pickle.load(f)

# PD frame for the whole portfolio (df_master_features is never modified).
# The development sample (matured loans) drives the EDA and the fitted treatment.
df_PD = df_master_features[["Target_Variable_Default", "is_matured"] + features_PD].copy()
dev_mask = df_PD["is_matured"] == 1
df_PD_dev = df_PD[dev_mask]

print(f"df_PD shape: {df_PD.shape} | development sample: {len(df_PD_dev):,} loans")
print(f"Features: {features_PD}")

## 2. Initial Audit

In [ ]:
audit_PD = dataset_audit(df_PD_dev[features_PD])
audit_PD

## 3. Numeric Features

### 3.1 Distribution Overview

In [ ]:
# Binary or categorical columns stored as numbers are one-hot encoded, not scaled
CATEGORICAL_NUMERIC = ["is_directpay_PD", "loan_term_months"]

num_features_PD = [
    f for f in df_PD[features_PD].select_dtypes(include=[np.number]).columns
    if f not in CATEGORICAL_NUMERIC
]
cat_features_PD = [f for f in features_PD if f not in num_features_PD]
print(f"Numeric: {num_features_PD}")
print(f"Categorical: {cat_features_PD}")

plot_boxplots_list(df_PD_dev, num_features_PD, cols=4)

### 3.2 Outlier Summary

In [ ]:
outlier_summary_PD = outlier_summary(df_PD_dev, num_features_PD)
outlier_summary_PD

### 3.3 Treatment Decisions

The treatment of every feature the selection can return is declared once, in `PD_TREATMENTS` (`src/clean_features_PD.py`), so the code follows the selection instead of hard-coding it:

- income: cap at p99, then log1p if the skew is still above 2;
- utilisation ratios: economic ceiling at 100;
- counts and balances with long upper tails: cap at p99 (log1p discarded for balances, where the mass near zero makes it invert the skew);
- debt-to-income: negatives set to 0 (data errors);
- credit history start year: floor at p1 (lower-tail errors);
- home ownership: OTHER (0.07%) grouped into RENT, the closest economic profile;
- remaining numeric NaNs: median of the development sample.

Every parameter (quantiles, the log1p decision, medians) is fitted on the development sample and applied unchanged to the whole portfolio. The table below lists what was applied to the features selected in this run.

In [ ]:
treatment_PD = fit_treatment_PD(df_PD_dev[features_PD])
df_PD_clean = df_PD[["Target_Variable_Default", "is_matured"]].join(
    treatment_PD.transform(df_PD[features_PD])
)
treatment_PD.summary()

In [ ]:
# Post-treatment audit (development sample)
audit_PD_num_clean = dataset_audit(df_PD_clean.loc[dev_mask, num_features_PD])
audit_PD_num_clean

In [ ]:
# Post-treatment boxplots (development sample)
plot_boxplots_list(df_PD_clean[dev_mask], num_features_PD, cols=4)

In [ ]:
# Post-treatment outlier summary (development sample)
outlier_summary_PD_num_clean = outlier_summary(df_PD_clean[dev_mask], num_features_PD)
outlier_summary_PD_num_clean

## 4. Categorical Features

In [ ]:
plot_categoricals(df_PD_clean[dev_mask], cat_features_PD, figsize_per_plot=(6, 4))

In [ ]:
# Category shares after the recoding in PD_TREATMENTS (development sample)
for col in cat_features_PD:
    print(df_PD_clean.loc[dev_mask, col].value_counts(normalize=True).round(4), "\n")

## 5. Post-Treatment Correlation Check

In [ ]:
corr_matrix_PD = compute_plot_correlation_matrix(df_PD_clean.loc[dev_mask, features_PD])

# Pairs still above the 0.75 threshold used in 02_feature_selection
pairs = corr_matrix_PD.where(np.triu(np.ones(corr_matrix_PD.shape, dtype=bool), k=1)).stack()
print("Pairs above 0.75 after treatment:")
print(pairs[pairs.abs() > 0.75].round(3) if (pairs.abs() > 0.75).any() else "none")

In [ ]:
# df_PD_clean holds the treated features for every loan; 05 trains on the matured ones
print(f"df_PD_clean: {df_PD_clean.shape} | matured: {int(df_PD_clean['is_matured'].sum()):,}")

## 6. Checkpoint: Save PD Training Dataframe

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/checkpoint_df_PD_training.pkl", "wb") as f:
    pickle.dump({"df_PD_clean": df_PD_clean}, f)

with open("data/processed/features_PD_num_for_training.pkl", "wb") as f:
    pickle.dump(num_features_PD, f)

with open("data/processed/features_PD_cat_for_training.pkl", "wb") as f:
    pickle.dump(cat_features_PD, f)

print(f"Saved. Shape: {df_PD_clean.shape}")

In [ ]:
# Load checkpoint (skip re-running)
with open("data/processed/checkpoint_df_PD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_PD_clean = data["df_PD_clean"]

with open("data/processed/features_PD_num_for_training.pkl", "rb") as f:
    num_features_PD = pickle.load(f)

with open("data/processed/features_PD_cat_for_training.pkl", "rb") as f:
    cat_features_PD = pickle.load(f)